# E3 schema-e3-v2.0 shard (rerun for the interval score)

Pre-filled for **DGP2**, training **n=500**, mode
**confirmatory**, seeds **0..99**. Run
cells from top to bottom; there is nothing to edit.

This is the same cell, the same seeds and the same models as the v1 run.
The driver is `run_cell_v2.R`, which adds the Winkler interval score at
the 50% and 95% levels (unit-level CATE and ATE, per model and outcome)
plus its non-coverage penalty component. Everything else is byte-
identical v1 code, and the added quantities consume no random numbers.

Two products, therefore. First, the interval score, which is the proper
scoring rule the model confidence set needs for the uncertainty
comparison and which answers the reviewer's point that coverage has a
target rather than a direction. Second, and free: the final cell checks
this shard against the published v1 CSV
`E3_DGP2_n500_confirmatory_shard01_seeds000-099.csv`, seed by seed and column by column. The
RNG state hashes and every v1 metric must match. A different machine,
a different session and a different day reproducing the numbers exactly
is the replication evidence the original study, which seeded from
`as.numeric(Sys.time())`, could not provide.


In [ ]:
import os, platform, subprocess, time

def sh(cmd):
    p = subprocess.run(
        cmd, shell=True, capture_output=True, text=True,
        encoding="utf-8", errors="replace")
    return p.stdout.strip()

print("hostname:", platform.node() or "n/a")
print("os:", platform.platform())
print("nproc:", sh("nproc"))
print("cpu:", sh("grep -m1 -E 'model name' /proc/cpuinfo") or "n/a")
print(sh("free -g") or "RAM information unavailable")
print("mc.cores is fixed at 2; model fits are fixed to one thread.")


In [ ]:
import subprocess
p = subprocess.run(
    ["bash", "-lc", "apt-get -qq update >/dev/null && "
     "apt-get -qq install -y --no-install-recommends "
     "r-base r-base-dev libcurl4-openssl-dev >/dev/null 2>&1"],
    capture_output=True, text=True, encoding="utf-8", errors="replace")
if p.returncode != 0:
    print(p.stdout[-1000:])
    print(p.stderr[-2000:])
    raise RuntimeError("R installation failed")
print(subprocess.check_output(
    ["R", "--version"], text=True,
    encoding="utf-8", errors="replace").splitlines()[0])


In [ ]:
import hashlib, os, re, shutil, subprocess, sys
from pathlib import Path

BUNDLE_FOLDER_URL = 'https://drive.google.com/drive/folders/1w3quuskj25CBOFCGG0mTRGUHcufPpdb3?usp=sharing'
BUNDLE_SHA256 = '12d223bc0fcef624c1ff4cc35c5d7ecc1b1f9b05aa84ecd9d9e4a5a3382bae3c'
assert re.fullmatch(r"[0-9a-fA-F]{64}", BUNDLE_SHA256), \
    "Bundle SHA256 is missing or malformed; regenerate the notebook."

BUNDLE_DOWNLOAD_DIR = Path("/content/tisca_bundle_download")
if BUNDLE_DOWNLOAD_DIR.exists():
    shutil.rmtree(BUNDLE_DOWNLOAD_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "gdown"],
    check=True,
)
# NOT `download`: that name is the Colab-download helper defined in the
# setup cell, and binding a CompletedProcess to it here made the final
# cell of E3_seed_verification.ipynb die with "'CompletedProcess' object
# is not callable" after a 50-minute run had already succeeded.
gdown_proc = subprocess.run(
    [sys.executable, "-m", "gdown", "--folder", BUNDLE_FOLDER_URL,
     "--output", str(BUNDLE_DOWNLOAD_DIR), "--remaining-ok"],
    capture_output=True, text=True,
    encoding="utf-8", errors="replace",
)
print(gdown_proc.stdout[-4000:])
if gdown_proc.returncode != 0:
    print(gdown_proc.stderr[-4000:])
    raise RuntimeError("Google Drive bundle download failed")
tar_candidates = sorted(BUNDLE_DOWNLOAD_DIR.rglob("tisca_rlib.tar.gz"))
sha_candidates = sorted(BUNDLE_DOWNLOAD_DIR.rglob("tisca_rlib.sha256"))
assert len(tar_candidates) == 1, f"expected one tarball, found {tar_candidates}"
assert len(sha_candidates) <= 1, f"expected at most one checksum file, found {sha_candidates}"
if sha_candidates:
    published_sha = sha_candidates[0].read_text().split()[0].lower()
    assert published_sha == BUNDLE_SHA256.lower(), \
        "published tisca_rlib.sha256 differs from the generated notebook"
    print("verified published checksum sidecar:", sha_candidates[0])
else:
    print("no tisca_rlib.sha256 sidecar in the shared folder; "
          "verifying the tarball against the embedded SHA256")
download_path = "/content/_dl_tisca_rlib.tar.gz"
shutil.copy2(tar_candidates[0], download_path)
with open(download_path, "rb") as f:
    observed_sha = hashlib.sha256(f.read()).hexdigest()
assert observed_sha == BUNDLE_SHA256.lower(), "R library bundle SHA mismatch"
if os.path.isdir("/content/tisca_rlib"):
    shutil.rmtree("/content/tisca_rlib")
subprocess.run(["tar", "xzf", download_path, "-C", "/content"], check=True)
LIBDIR = "/content/tisca_rlib/rlib"
assert os.path.isdir(LIBDIR), "bundle did not restore the expected rlib directory"
print("bundle restored:", LIBDIR, "from", BUNDLE_FOLDER_URL)


In [ ]:
import os, urllib.request

RUNCELL_URL = (
    "https://raw.githubusercontent.com/hugogobato/"
    "Test-Informed-Simulation-Count-Algorithm-TISCA/main/"
    "experiments/E3_mvbcf_casestudy/run_cell_v2.R"
)
MVBCF_CPP_URL = (
    "https://raw.githubusercontent.com/Nathan-McJames/MVBCF_Paper/"
    "main/MVBCF_Code.cpp"
)
os.makedirs("/content/e3", exist_ok=True)
urllib.request.urlretrieve(RUNCELL_URL, "/content/e3/run_cell_v2.R")
# The upstream C++ is downloaded at runtime and is never committed here.
urllib.request.urlretrieve(MVBCF_CPP_URL, "/content/e3/MVBCF_Code.cpp")
assert os.path.getsize("/content/e3/run_cell_v2.R") > 1000
assert os.path.getsize("/content/e3/MVBCF_Code.cpp") > 10000
with open("/content/e3/run_cell_v2.R") as f:
    run_cell_source = f.read()
required_fixes = [
    # v1 invariants: these must still be exactly as they were, because
    # the parity check below assumes the fitting code is unchanged.
    "nthread = nthread_global",
    "num_threads = nthread_global",
    "acquired <- dir.create(lk",
    "num_gfr = 0",
    "sigma2_leaf_init = 1^2 / n_tree_mu",
    "sigma2_leaf_init = 0.375^2 / n_tree_tau",
    'propensity_covariate = "prognostic"',
    "sample_sigma2_leaf = FALSE",
    # v2 additions (P5.5-T1).
    "E3_SCHEMA_VERSION",
    '"e3-v2.0"',
    "interval_score_mat <- function",
    "interval_score_vec <- function",
    "fill_interval_scores <- function",
    "width_audit_max_abs_dev",
]
missing_fixes = [item for item in required_fixes if item not in run_cell_source]
assert not missing_fixes, (
    "GitHub main is serving a driver without the schema-e3-v2.0 "
    "interval-score code. Commit and push run_cell_v2.R before running "
    f"this notebook; missing: {missing_fixes}"
)
print("downloaded run_cell_v2.R (schema e3-v2.0) and upstream MVBCF_Code.cpp")


In [ ]:
import os, subprocess

compile_script = "\n".join([
    ".libPaths(c('/content/tisca_rlib/rlib', .libPaths()))",
    "if (!requireNamespace('Rcpp', quietly=TRUE) ||",
    "    !requireNamespace('RcppArmadillo', quietly=TRUE) ||",
    "    !requireNamespace('RcppDist', quietly=TRUE)) stop('bundle missing Rcpp dependencies')",
    "library(Rcpp)",
    "sourceCpp('/content/e3/MVBCF_Code.cpp')",
    "stopifnot(is.function(fast_bart))",
    "cat('FAST_BART_OK\\n')",
])
with open("/content/e3/compile.R", "w") as f:
    f.write(compile_script)
p = subprocess.run(["Rscript", "/content/e3/compile.R"],
                   capture_output=True, text=True,
                   encoding="utf-8", errors="replace")
print(p.stdout[-3000:])
if p.returncode != 0 or "FAST_BART_OK" not in p.stdout:
    print(p.stderr[-3000:])
    raise RuntimeError("upstream MVBCF C++ compilation failed")
print("fast_bart() compiled")


    ## Fixed shard configuration

    The constants below were generated from `shard_table.csv`. They are
    assertions, not operator inputs. A dropped session can be restarted by
    uploading this same notebook. Successful seed rows are preserved, while
    failed or malformed checkpoint rows are backed up, removed, and retried.
    

In [ ]:
import csv, os, shutil, subprocess, time

DGP = 2
N = 500
MODE = 'confirmatory'
SHARD_ID = '01'
CLI_SEED_START = 0
CLI_SEED_END = 99
EXPECTED_SEED_START = 0
EXPECTED_SEED_END = 99
MC_CORES = 2
OUTPUT_DIR = "/content/TISCA_E3"
OUTPUT_CSV = os.path.join(OUTPUT_DIR, 'E3v2_DGP2_n500_confirmatory_shard01_seeds000-099.csv')
GIT_SHA = "not-a-git-build"

assert DGP in (1, 2, 3)
assert N in (100, 500)
assert MODE in ("pilot", "confirmatory")
assert MC_CORES == 2
assert CLI_SEED_START == 0 or CLI_SEED_START > 0
assert CLI_SEED_END >= CLI_SEED_START
assert EXPECTED_SEED_END - EXPECTED_SEED_START + 1 == CLI_SEED_END - CLI_SEED_START + 1
os.makedirs(OUTPUT_DIR, exist_ok=True)

def emitted_seed(raw_seed):
    return raw_seed + 1000001 if MODE == "pilot" else raw_seed

def contiguous_ranges(values):
    values = sorted(values)
    if not values:
        return []
    out = []
    start = previous = values[0]
    for value in values[1:]:
        if value != previous + 1:
            out.append((start, previous))
            start = value
        previous = value
    out.append((start, previous))
    return out

expected = {emitted_seed(i) for i in range(CLI_SEED_START, CLI_SEED_END + 1)}

def checkpoint_seed(row):
    raw = (row.get("seed") or "").strip()
    try:
        return int(raw)
    except (TypeError, ValueError):
        return None

existing = set()
if os.path.exists(OUTPUT_CSV):
    with open(OUTPUT_CSV, newline="") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        rows = list(reader)
    if not fieldnames or "seed" not in fieldnames or "converged_flag" not in fieldnames:
        raise RuntimeError("checkpoint CSV is missing required columns")

    kept_rows = []
    rejected = {"malformed": 0, "failed": 0, "duplicate": 0}
    for row in rows:
        seed = checkpoint_seed(row)
        if seed is None or None in row or any(value is None for value in row.values()):
            rejected["malformed"] += 1
            continue
        if seed not in expected:
            raise RuntimeError(
                f"checkpoint contains seed {seed} outside this shard; "
                "check that the correct CSV was uploaded"
            )
        if row.get("converged_flag") != "1":
            rejected["failed"] += 1
            continue
        if seed in existing:
            rejected["duplicate"] += 1
            continue
        existing.add(seed)
        kept_rows.append(row)

    if any(rejected.values()):
        backup = f"{OUTPUT_CSV}.pre_repair_{int(time.time())}.csv"
        shutil.copy2(OUTPUT_CSV, backup)
        temporary = OUTPUT_CSV + ".repair.tmp"
        with open(temporary, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(kept_rows)
        os.replace(temporary, OUTPUT_CSV)
        print("checkpoint repair:", rejected, "backup:", backup)

# A killed older run may leave either the former file lock or the new
# atomic lock directory behind. No R process is active at this point.
stale_lock = OUTPUT_CSV + ".lock"
if os.path.isdir(stale_lock):
    shutil.rmtree(stale_lock)
    print("removed stale checkpoint lock directory")
elif os.path.isfile(stale_lock):
    os.remove(stale_lock)
    print("removed stale checkpoint lock file")

missing_raw = [i for i in range(CLI_SEED_START, CLI_SEED_END + 1)
               if emitted_seed(i) not in existing]
print("checkpoint:", len(existing), "rows; missing:", len(missing_raw))
print("Local checkpoint CSV:", OUTPUT_CSV)


In [ ]:
# Run only missing contiguous ranges. This makes a re-upload idempotent
# while preserving run_cell_v2.R's shard-offset-invariant RNG streams.
env = dict(os.environ)
env["R_LIBS"] = LIBDIR + ":" + env.get("R_LIBS", "")
env["TISCA_MVBCF_CPP"] = "/content/e3/MVBCF_Code.cpp"
env["TISCA_GIT_SHA"] = GIT_SHA
for raw_start, raw_end in contiguous_ranges(missing_raw):
    cmd = ["Rscript", "/content/e3/run_cell_v2.R", str(DGP), str(N),
           str(raw_start), str(raw_end), "--out", OUTPUT_CSV,
           "--cores", str(MC_CORES), "--mode", MODE]
    print("running missing range:", " ".join(cmd))
    t0 = time.time()
    result = subprocess.run(
        cmd, env=env, capture_output=True, text=True,
        encoding="utf-8", errors="replace")
    print(result.stdout[-6000:])
    print("range wall-clock: %.1f min" % ((time.time() - t0) / 60.0))
    if result.returncode != 0:
        print(result.stderr[-4000:])
        raise RuntimeError("run_cell_v2.R failed")


In [ ]:
import csv, os
with open(OUTPUT_CSV, newline="") as f:
    rows = list(csv.DictReader(f))
malformed = [r.get("seed") for r in rows if checkpoint_seed(r) is None]
assert not malformed, f"checkpoint contains malformed seeds: {{malformed[:5]}}"
got = [checkpoint_seed(r) for r in rows]
expected = list(range(EXPECTED_SEED_START, EXPECTED_SEED_END + 1))
duplicates = len(got) - len(set(got))
missing = sorted(set(expected) - set(got))
unexpected = sorted(set(got) - set(expected))
failures = [r for r in rows if r.get("converged_flag") != "1"]
print("completed rows:", len(rows), "of", len(expected))
print("duplicate seeds:", duplicates, "missing:", len(missing),
      "unexpected:", len(unexpected))
print("converged_flag failures:", len(failures))
print("checkpoint bytes:", os.path.getsize(OUTPUT_CSV))
try:
    from google.colab import files
    files.download(OUTPUT_CSV)
    print("Downloaded:", OUTPUT_CSV)
except Exception as e:
    print("(Not on Colab / download skipped):", e)
assert duplicates == 0, "shard checkpoint contains duplicate seeds"
assert not unexpected, f"checkpoint has seeds outside this shard: {{unexpected[:5]}}"
assert not missing, f"shard checkpoint is incomplete; missing seeds: {{missing[:10]}}"
assert not failures, (
    f"{{len(failures)}} replications failed; rerun the notebook to retry them. "
    f"First errors: {{[r.get('error_message') for r in failures[:3]]}}"
)


## v1 parity check (this is the replication evidence)

Downloads the v1 CSV for these exact seeds and compares it with what this
session just produced. Three groups of columns are treated differently.

**RNG state and model seeds** must be identical strings. These are the
L'Ecuyer substream states and the integer seeds drawn from them, so a match
proves both runs sampled from the same streams in the same order.

**v1 metrics** must be identical to floating-point noise. They are computed
by unchanged code from those streams.

**Provenance and timing** (`hostname`, `git_sha`, `session_hash`,
`fit_seconds_*`, `replication_seconds`) are *expected* to differ: this is a
different machine on a different day. They are reported, never asserted.

The verdict is printed and written to a JSON file, and it is not fatal. A
long fit should not be discarded by a Colab cell; whether a deviation
matters is an analysis decision, made once against all 33 shards.


In [ ]:
import csv, json, math, os, urllib.request

V1_URL = 'https://raw.githubusercontent.com/hugogobato/Test-Informed-Simulation-Count-Algorithm-TISCA/main/notebooks/E3_shards/E3_DGP2_n500_confirmatory_shard01_seeds000-099.csv'
V1_LOCAL = "/content/TISCA_E3/v1_" + 'E3_DGP2_n500_confirmatory_shard01_seeds000-099.csv'
PARITY_JSON = OUTPUT_CSV.replace(".csv", "_parity.json")

STREAM_EXACT = ["rng_kind", "seed_data_hash", "seed_fit_hash",
                "seed_cell_master", "seq_phase", "n", "dgp",
                "model_seed_propensity", "model_seed_mvbcf",
                "model_seed_bcf1", "model_seed_bcf2",
                "model_seed_bart1", "model_seed_bart2",
                "model_seed_mvbart"]
PROVENANCE = ["hostname", "git_sha", "session_hash", "error_message",
              "replication_seconds", "fit_seconds_mvbcf",
              "fit_seconds_bcf1", "fit_seconds_bcf2",
              "fit_seconds_bart1", "fit_seconds_bart2",
              "fit_seconds_mvbart"]

urllib.request.urlretrieve(V1_URL, V1_LOCAL)
with open(V1_LOCAL, newline="") as f:
    v1_rows = {int(r["seed"]): r for r in csv.DictReader(f)}
with open(OUTPUT_CSV, newline="") as f:
    v2_reader = csv.DictReader(f)
    v2_fields = v2_reader.fieldnames
    v2_rows = {int(r["seed"]): r for r in v2_reader}

v1_fields = list(next(iter(v1_rows.values())).keys())
new_columns = [c for c in v2_fields if c not in v1_fields]
dropped_columns = [c for c in v1_fields if c not in v2_fields]
shared_seeds = sorted(set(v1_rows) & set(v2_rows))

# v2 must be a strict superset of the v1 schema, in v1's order.
prefix_ok = v2_fields[:len(v1_fields)] == v1_fields

stream_mismatch = {}
for column in STREAM_EXACT:
    if column not in v1_fields or column not in v2_fields:
        continue
    bad = [s for s in shared_seeds
           if v1_rows[s][column] != v2_rows[s][column]]
    if bad:
        stream_mismatch[column] = len(bad)

compare = [c for c in v1_fields
           if c in v2_fields and c not in STREAM_EXACT
           and c not in PROVENANCE and c != "seed"]
worst = {"column": None, "seed": None, "abs": 0.0, "rel": 0.0}
exact = 0
nonnumeric = []
for column in compare:
    for seed in shared_seeds:
        a, b = v1_rows[seed][column], v2_rows[seed][column]
        if a == b:
            exact += 1
            continue
        try:
            x, y = float(a), float(b)
        except (TypeError, ValueError):
            nonnumeric.append((column, seed))
            continue
        d = abs(x - y)
        r = d / max(abs(x), 1e-12)
        if r > worst["rel"]:
            worst = {"column": column, "seed": seed, "abs": d, "rel": r}

total = len(compare) * len(shared_seeds)
if stream_mismatch or nonnumeric or not prefix_ok:
    verdict = "MISMATCH"
elif exact == total:
    verdict = "BIT_IDENTICAL"
elif worst["rel"] < 1e-12:
    verdict = "IDENTICAL_TO_FP_NOISE"
elif worst["rel"] < 1e-6:
    verdict = "AGREES_LOOSELY"
else:
    verdict = "MISMATCH"

report = {
    "shard": os.path.basename(OUTPUT_CSV),
    "v1_shard": 'E3_DGP2_n500_confirmatory_shard01_seeds000-099.csv',
    "schema_version": 'e3-v2.0',
    "verdict": verdict,
    "seeds_compared": len(shared_seeds),
    "v1_only_seeds": sorted(set(v1_rows) - set(v2_rows)),
    "v2_only_seeds": sorted(set(v2_rows) - set(v1_rows)),
    "v1_schema_is_prefix_of_v2": prefix_ok,
    "new_columns": new_columns,
    "dropped_columns": dropped_columns,
    "compared_cells": total,
    "bit_identical_cells": exact,
    "stream_mismatch": stream_mismatch,
    "nonnumeric_mismatch": nonnumeric[:20],
    "worst_relative_deviation": worst,
}
with open(PARITY_JSON, "w") as f:
    json.dump(report, f, indent=2)

print("v1 parity verdict:", verdict)
print("seeds compared:", len(shared_seeds),
      "| cells compared:", total,
      "| bit-identical:", exact)
print("new v2 columns:", len(new_columns))
print("v1 schema is a prefix of v2:", prefix_ok)
if dropped_columns:
    print("WARNING dropped v1 columns:", dropped_columns)
if stream_mismatch:
    print("WARNING RNG/seed columns differ:", stream_mismatch)
if worst["column"]:
    print("worst deviation: %s seed %s abs=%.3g rel=%.3g"
          % (worst["column"], worst["seed"], worst["abs"], worst["rel"]))
try:
    from google.colab import files
    files.download(PARITY_JSON)
    print("Downloaded:", PARITY_JSON)
except Exception as e:
    print("(Not on Colab / download skipped):", e)


## Interval-score sanity summary

A cheap look at what the rerun bought, before any of it reaches an
analysis. The width audit compares the interval score's own width component
against the v1 `wid` columns; the driver already fails a replication whose
disagreement exceeds `1e-8` times the width scale, so this should print
zero. The score identity `is = wid + pen` is checked here in the collected
file rather than trusted from the driver.


In [ ]:
import csv, statistics

with open(OUTPUT_CSV, newline="") as f:
    rows = list(csv.DictReader(f))

audit = [float(r["width_audit_max_abs_dev"]) for r in rows]
print("width audit, max over replications: %.3g" % max(audit))

worst_identity = 0.0
for model in ("mvbcf", "bcf", "bart", "mvbart"):
    for level in ("50", "95"):
        for k in ("1", "2"):
            for r in rows:
                score = float(r[f"{model}_is{level}{k}"])
                parts = (float(r[f"{model}_wid{level}{k}"])
                         + float(r[f"{model}_pen{level}{k}"]))
                worst_identity = max(
                    worst_identity,
                    abs(score - parts) / max(abs(score), 1e-12))
print("max relative violation of is == wid + pen: %.3g" % worst_identity)

print()
print("%-8s %-6s %-10s %-10s %-10s %-10s"
      % ("model", "level", "IS(mean)", "width", "penalty", "coverage"))
for model in ("mvbcf", "bcf", "bart", "mvbart"):
    for level in ("50", "95"):
        for k in ("1", "2"):
            def mean_of(prefix):
                return statistics.mean(
                    float(r[f"{model}_{prefix}{level}{k}"]) for r in rows)
            print("%-8s %-6s %-10.4f %-10.4f %-10.4f %-10.4f"
                  % (f"{model}.y{k}", level + "%", mean_of("is"),
                     mean_of("wid"), mean_of("pen"), mean_of("cov")))
print()
print("Lower interval score is better. It is NOT the same ranking as "
      "coverage: a model can cover well by being wide and still score "
      "badly, which is the point of recording it.")
